# RExA — DistilBERT Star Prediction (Google Colab)

Fine-tune **`distilbert-base-uncased`** on ASAP 2.0 + ASAP-SAS (AERA).

**Before running:**
1. Runtime → Change runtime type → **GPU (T4)** → Save
2. Runtime → **Run all**

**No file upload needed.** Datasets download from Hugging Face automatically.

At the end you get `distilbert_stars.zip` to download into your PC.

## 1. Install packages + check GPU

In [ ]:
import sys, subprocess
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.44.0,<5.0',
    'datasets>=2.19',
    'accelerate>=0.30',
    'evaluate',
    'scikit-learn',
    'scipy',
    'pandas',
])

import torch
import transformers
print('transformers', transformers.__version__)
print('torch', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU. Go to Runtime → Change runtime type → GPU, then re-run.')

## 2. Helpers (dataset load + metrics)\nNo upload dialog in this notebook.

In [ ]:
import json, random, inspect
from pathlib import Path
import numpy as np
import torch
from datasets import Dataset, load_dataset
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import spearmanr
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

def map_score(score, lo, hi):
    norm = (float(score) - lo) / max(hi - lo, 1e-9)
    return float(np.clip(1.0 + 4.0 * norm, 1.0, 5.0))

def load_corpus():
    rows = []
    print('Loading ASAP 2.0 from Hugging Face...')
    aes = load_dataset('jatinmehra/Automated-Essay-Scoring-2.0')
    split = list(aes.keys())[0]
    for item in aes[split]:
        text = (item.get('full_text') or '').strip()
        if len(text) < 40:
            continue
        rows.append({
            'text': text[:4000],
            'stars': map_score(item['score'], 1, 6),
            'source': 'asap2',
        })
    print('ASAP2 rows:', len(rows))

    print('Loading AERA / ASAP-SAS...')
    try:
        aera = load_dataset(
            'jiazhengli/AERA',
            data_files={
                'train': 'simple/train.json',
                'val': 'simple/val.json',
                'test': 'simple/test.json',
            },
        )
        before = len(rows)
        for split_name in aera:
            for item in aera[split_name]:
                text = (item.get('EssayText') or '').strip()
                if len(text) < 20 or item.get('Score1') is None:
                    continue
                rows.append({
                    'text': text[:4000],
                    'stars': map_score(item['Score1'], 0, 3),
                    'source': 'aera',
                })
        print('AERA added:', len(rows) - before)
    except Exception as e:
        print('AERA skipped (ASAP2 alone is enough):', e)

    print('TOTAL rows:', len(rows))
    return rows

def split_rows(rows, seed=42):
    rng = random.Random(seed)
    buckets = {}
    for r in rows:
        buckets.setdefault(int(round(r['stars'])), []).append(r)
    train, val, test = [], [], []
    for b in buckets.values():
        rng.shuffle(b)
        n = len(b)
        nt, nv = max(1, int(0.15 * n)), max(1, int(0.10 * n))
        test += b[:nt]
        val += b[nt:nt + nv]
        train += b[nt + nv:]
    rng.shuffle(train); rng.shuffle(val); rng.shuffle(test)
    return train, val, test

def star_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.clip(np.asarray(y_pred, dtype=float), 1, 5)
    mae = float(mean_absolute_error(y_true, y_pred))
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2 = float(r2_score(y_true, y_pred))
    rho, p = spearmanr(y_true, y_pred)
    within = float(np.mean(np.abs(np.rint(y_true) - np.rint(y_pred)) <= 1))
    exact = float(np.mean(np.rint(y_true) == np.rint(y_pred)))
    return {
        'mae': round(mae, 4),
        'rmse': round(rmse, 4),
        'r2': round(r2, 4),
        'spearman_rho': round(float(rho), 4),
        'spearman_pvalue': round(float(p), 4),
        'within_one_star_accuracy': round(within, 4),
        'exact_accuracy': round(exact, 4),
        'n_samples': int(len(y_true)),
    }

def make_training_args(output_dir, batch, epochs, lr):
    """Compatible with older and newer transformers versions."""
    kwargs = {
        'output_dir': str(output_dir),
        'learning_rate': lr,
        'per_device_train_batch_size': batch,
        'per_device_eval_batch_size': batch,
        'num_train_epochs': epochs,
        'weight_decay': 0.01,
        'warmup_ratio': 0.06,
        'save_strategy': 'epoch',
        'load_best_model_at_end': True,
        'metric_for_best_model': 'mae',
        'greater_is_better': False,
        'fp16': torch.cuda.is_available(),
        'logging_steps': 50,
        'report_to': [],
        'remove_unused_columns': False,
    }
    params = inspect.signature(TrainingArguments.__init__).parameters
    if 'eval_strategy' in params:
        kwargs['eval_strategy'] = 'epoch'
    else:
        kwargs['evaluation_strategy'] = 'epoch'
    return TrainingArguments(**kwargs)

print('Helpers ready')

## 3. Load datasets from Hugging Face

In [ ]:
rows = load_corpus()
assert len(rows) > 1000, 'Dataset too small — check internet / Hugging Face access'
train_rows, val_rows, test_rows = split_rows(rows)
print('train/val/test:', len(train_rows), len(val_rows), len(test_rows))

## 4. Fine-tune DistilBERT\nSet `SMOKE = True` for a quick test (~10 min). Use `SMOKE = False` for full FYP training.

In [ ]:
MODEL_NAME = 'distilbert-base-uncased'
MAX_LEN = 256
BATCH = 16 if torch.cuda.is_available() else 4
EPOCHS = 3
LR = 2e-5
SMOKE = False  # True = quick test; False = full training

if SMOKE:
    train_rows = train_rows[:800]
    val_rows = val_rows[:200]
    test_rows = test_rows[:200]
    EPOCHS = 1
    print('SMOKE MODE: small subset')

set_seed(42)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_ds(part):
    return Dataset.from_dict({
        'text': [r['text'] for r in part],
        'labels': [float(r['stars']) for r in part],
    })

def tok(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LEN)

train_ds = to_ds(train_rows).map(tok, batched=True, remove_columns=['text'])
val_ds = to_ds(val_rows).map(tok, batched=True, remove_columns=['text'])
test_ds = to_ds(test_rows).map(tok, batched=True, remove_columns=['text'])

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=1,
    problem_type='regression',
)

out_dir = Path('/content/distilbert_stars')
out_dir.mkdir(parents=True, exist_ok=True)
args = make_training_args(out_dir / 'runs', BATCH, EPOCHS, LR)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.asarray(logits).reshape(-1)
    labels = np.asarray(labels).reshape(-1)
    m = star_metrics(labels, preds)
    return {
        'mae': m['mae'],
        'spearman': m['spearman_rho'],
        'within_one': m['within_one_star_accuracy'],
    }

trainer_kwargs = dict(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)
# transformers API: tokenizer= is deprecated in newer versions
trainer_params = inspect.signature(Trainer.__init__).parameters
if 'processing_class' in trainer_params:
    trainer_kwargs['processing_class'] = tokenizer
else:
    trainer_kwargs['tokenizer'] = tokenizer

trainer = Trainer(**trainer_kwargs)
print('Starting training...')
trainer.train()
print('Training finished.')

## 5. Evaluate + save model

In [ ]:
pred_out = trainer.predict(test_ds)
preds = np.asarray(pred_out.predictions).reshape(-1)
labels = np.asarray(pred_out.label_ids).reshape(-1)
metrics = star_metrics(labels, preds)
print('TEST METRICS:')
print(json.dumps(metrics, indent=2))

model_dir = out_dir / 'model'
trainer.save_model(str(model_dir))
tokenizer.save_pretrained(str(model_dir))

payload = {
    'model_name': MODEL_NAME,
    'train_samples': len(train_rows),
    'val_samples': len(val_rows),
    'test_samples': len(test_rows),
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'test': metrics,
}
(out_dir / 'metrics.json').write_text(json.dumps(payload, indent=2))
print('Saved model to', model_dir)
print('Saved metrics to', out_dir / 'metrics.json')
print('Files:', [p.name for p in model_dir.iterdir()])

## 6. Create zip + download\nThis creates **`distilbert_stars.zip`**. Download that file (not the .ipynb).

In [ ]:
import shutil
from google.colab import files

assert (out_dir / 'model').exists(), 'Model folder missing — re-run cell 5'
assert (out_dir / 'metrics.json').exists(), 'metrics.json missing — re-run cell 5'

zip_path = '/content/distilbert_stars.zip'
# zip the contents of out_dir
shutil.make_archive('/content/distilbert_stars', 'zip', root_dir=str(out_dir))
print('Created:', zip_path)
print('Size MB:', round(Path(zip_path).stat().st_size / 1e6, 2))

files.download(zip_path)
print('Download started. Check your PC Downloads for distilbert_stars.zip')

## Done\nOn your PC you should have **`distilbert_stars.zip`** (large file, usually 200MB+).\nTell your Cursor agent “done” and they will install it into the RExA project.\n\nIf download fails: left sidebar Files → refresh → right-click `distilbert_stars.zip` → Download.